In [1]:
import numpy as np
import pandas as pd
import os
from src.data_loader import prepare_paths, load_horizontal_well, load_typewell
from src.features import (generate_signal_features, generate_spatial_features,
        integrate_with_typewells, full_preprocessing_pipeline)
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from src.prepare_data import prepare_target
from src.prepare_data import train_test_split_paths
from sklearn.metrics import root_mean_squared_error
from src.model_utils import validate_model

In [2]:
model_xgb = xgb.XGBRegressor()
model_xgb.load_model('xgb_model.json')

In [3]:
paths_test_str = 'test'
paths_test = prepare_paths(paths_test_str)

In [4]:
test = []

for path in paths_test:
    new_df = full_preprocessing_pipeline(*path)
    test.append(new_df)

/Users/iaroslav/Downloads/wellbore-geology-prediction/src/features.py:19: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[f'gr_lag_{lag}'] = df['GR'].shift(lag).fillna(method='bfill')
/Users/iaroslav/Downloads/wellbore-geology-prediction/src/features.py:20: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[f'gr_lead_{lag}'] = df['GR'].shift(-lag).fillna(method='ffill')
/Users/iaroslav/Downloads/wellbore-geology-prediction/src/features.py:19: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[f'gr_lag_{lag}'] = df['GR'].shift(lag).fillna(method='bfill')
/Users/iaroslav/Downloads/wellbore-geology-prediction/src/features.py:20: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.f

In [5]:
test_concated = pd.concat(test)
test_concated.head()

test_concated["TVT_prev"] = 0
for row in range(len(test_concated)):
    prev_tvt = test_concated.iloc[max(0, row-1)]["TVT_input"]
    if prev_tvt == np.nan:
        prev_tvt = test_concated["TVT_prev"]
    test_concated.iloc[row]["TVT_prev"] = prev_tvt


/var/folders/sl/c1q8vzl53xl0g5kgw_r8t_bh0000gn/T/ipykernel_35840/3009782695.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_concated.iloc[row]["TVT_prev"] = prev_tvt


In [10]:
tvt_input = test_concated["TVT_input"]
test_no_input = test_concated.drop(columns=["TVT_input"])

predictions = model_xgb.predict(test_no_input)

In [12]:
pred_input = pd.DataFrame(predictions, tvt_input)
pred_input.head()

,0
TVT_input,
11236.02,10192.409180
11237.05,10152.499023
11238.09,10156.683594
11239.12,10159.272461
11240.15,10157.411133


In [ ]:
path_df = {}

for i in range(len(paths_test)):
    horizontal_path, _ = paths_test[i]
    path_df[horizontal_path] = test[i]

total_rows = 0

output_rows = []
for horizontal_path, value in path_df.items():

    well_id = os.path.basename(horizontal_path).replace("__horizontal_well.csv", "")
    for row_idx in range(len(value)):
        output_rows.append({
            "id": f"{well_id}_{row_idx}",
            "tvt": float(predictions[total_rows])
        })
        total_rows += 1


output = pd.DataFrame(output_rows)
output.head()


,id,tvt
0,000d7d20_0,10192.409180
1,000d7d20_1,10152.499023
2,000d7d20_2,10156.683594
3,000d7d20_3,10159.272461
4,000d7d20_4,10157.411133


In [8]:
output.to_csv("submission.csv", index=False)